# Übungen – Block 12: Praxisprojekt

Dieses Notebook beschreibt das Abschlussprojekt. Die konkreten Hostnamen und Dienste werden an die Schulungsumgebung angepasst.

## Musterlösung

Die Lösung ist bewusst als Referenzarchitektur formuliert. Es sind mehrere gleichwertige Projektvarianten möglich.

## Projektauftrag

Automatisieren Sie eine kleine Linux-Serverlandschaft mit mindestens zwei Hostgruppen.

Der Gesamtprozess soll Linux, Python und Ansible sinnvoll kombinieren.

## Teil 1: Architektur und Verantwortlichkeiten

Dokumentieren Sie:

- Control Node
- Managed Nodes
- Hostgruppen
- SSH-Zugriff
- welche Aufgabe Linux, Shell, Python und Ansible jeweils übernehmen.

```text
Control Node: admin01
Managed Nodes: web01, web02, db01
Gruppen: web, db

Shell: optionaler Startablauf
Python: Datenvalidierung und Ableitung
Ansible: Paket, Datei, Dienst, Konfiguration
```

## Teil 2: Python-Komponente

Erstellen Sie ein Python-Programm, das eine JSON-Datei mit Server- oder Anwendungsdaten einliest, Pflichtfelder prüft und daraus eine neue JSON-Datei mit abgeleiteten Konfigurationswerten erzeugt.

```python
import json
from pathlib import Path

src = Path("servers.json")
dst = Path("generated.json")

servers = json.loads(src.read_text(encoding="utf-8"))
result = []

for server in servers:
    if "name" not in server or "role" not in server:
        raise ValueError("name und role sind Pflichtfelder")

    item = dict(server)
    item["service_port"] = 8080 if server["role"] == "web" else 5432
    result.append(item)

dst.write_text(json.dumps(result, indent=2), encoding="utf-8")
```

## Teil 3: Ansible-Projekt

Erstellen Sie:

- Inventory
- `group_vars` bzw. `host_vars`
- mindestens eine Rolle
- Template
- Handler
- mindestens einen factabhängigen Task

Die Rolle soll einen Dienst bzw. ein freigegebenes Paket verwalten und eine Konfigurationsdatei erzeugen.

```text
ansible/
├── inventory.yml
├── group_vars/
│   ├── web.yml
│   └── db.yml
├── host_vars/
├── roles/
│   └── app_config/
│       ├── defaults/main.yml
│       ├── handlers/main.yml
│       ├── tasks/main.yml
│       └── templates/app.conf.j2
└── site.yml
```

`site.yml`:

```yaml
- hosts: all
  become: true
  roles:
    - app_config
```

## Teil 4: Integration

Binden Sie die vom Python-Programm erzeugten Daten in den Gesamtprozess ein. Entscheiden Sie, ob ein einfaches Shell-Skript die Aufrufe nacheinander orchestrieren soll.

Optionales Shell-Skript:

```bash
#!/bin/bash
set -e

python3 generate_config.py
ansible all -i ansible/inventory.yml -m ping
ansible-playbook -i ansible/inventory.yml ansible/site.yml
```

Die Shell koordiniert nur vorhandene Werkzeuge; die Programmlogik bleibt in Python, die Systemzustände in Ansible.

## Teil 5: Test

1. Führen Sie Connectivity Checks aus.
2. Starten Sie den ersten vollständigen Run.
3. Prüfen Sie Zielzustände.
4. Führen Sie denselben Run erneut aus.
5. Dokumentieren Sie alle noch auftretenden `changed`-Tasks.
6. Erzeugen Sie einen kontrollierten Fehlerfall und analysieren Sie ihn.

Erwartetes Testvorgehen:

```bash
ansible all -i ansible/inventory.yml -m ping
ansible-playbook -i ansible/inventory.yml ansible/site.yml
ansible-playbook -i ansible/inventory.yml ansible/site.yml
```

Der zweite Run sollte bei unverändertem Zielzustand möglichst keine Änderungen mehr melden.

Für den Fehlerfall kann beispielsweise ein falscher Hostname oder eine ungültige Variable bewusst gesetzt und anschließend mit `-vv` sowie `debug` untersucht werden.

## Abschlussreflexion

Beantworten Sie:

1. Welche Aufgaben waren mit Shell sinnvoll?
2. Welche Logik gehörte nach Python?
3. Welche Aufgaben waren typische Ansible-Zustandsverwaltung?
4. Wo spielte Idempotenz eine Rolle?
5. Wie würde sich die Lösung bei 100 Hosts verändern?
6. Welche Teile könnten als nächstes professionalisiert werden?

## Musterantwort zur Abschlussreflexion

1. Shell eignet sich für einen einfachen Aufrufablauf mehrerer vorhandener Programme.
2. Validierung, Datenverarbeitung und eigene Berechnungslogik gehören in Python.
3. Paketinstallation, Dateiverteilung, Konfiguration und Dienstzustände sind typische Ansible-Aufgaben.
4. Idempotenz zeigt sich vor allem beim wiederholten Ansible-Run.
5. Bei 100 Hosts bleiben Rollen und Inventories grundsätzlich tragfähig; Gruppenstruktur, Fehlermanagement, Parallelisierung und sichere Secrets-Verwaltung gewinnen an Bedeutung.
6. Nächste Professionalisierungsschritte wären beispielsweise Vault/Secrets, Collections, CI-Tests, Linting, Molecule oder zentrale Automation-Plattformen.


## Vertiefte Musterantworten zu Integration, Tests und Reflexion

### Teil 1 – Verantwortlichkeiten
Der Control Node hält Ansible, Inventory und die Projektdateien; Managed Nodes stellen die verwalteten Zielsysteme dar. Linux liefert Betriebssystem, SSH und Dienste. Shell kann die Reihenfolge weniger lokaler Befehle koordinieren. Python validiert Eingangsdaten und berechnet Konfigurationen. Ansible setzt die gewünschten Zustände auf mehreren Hosts um. Zugangsdaten gehören nicht in das Inventory im Klartext.

### Teil 2 – Python-Validierung
Neben `name` und `role` sollten Datentypen, leere Namen, erlaubte Rollen und doppelte Hostnamen geprüft werden. Bei fehlerhaften Eingaben muss das Programm mit verständlicher Fehlermeldung abbrechen, bevor eine unvollständige `generated.json` als gültige Konfiguration weiterverwendet wird. Ein einfacher Test umfasst gültige Daten, fehlende Pflichtfelder und falsche Datentypen.

### Teil 3 – Rolle und Konfiguration
Eine vollständige Lösung enthält ein Inventory mit mindestens zwei Gruppen, Gruppen- oder Hostvariablen, `roles/app_config/tasks/main.yml`, ein Template und einen Handler. Der Template-Task verwendet `notify`; der Handler wird nur bei einer Änderung ausgeführt. Ein factabhängiger Task benötigt ermittelte Facts und eine nachvollziehbare `when`-Bedingung. Das Paket und der Dienst müssen auf den Schulungshosts tatsächlich verfügbar sein.

### Teil 4 – Datenfluss von Python zu Ansible
Das bloße Ausführen von `generate_config.py` vor dem Playbook genügt nicht: Das Playbook muss die erzeugten Werte tatsächlich einlesen. Eine mögliche Variante ist, `generated.json` als Ansible-Variablendatei mit einer Dictionary-Struktur zu erzeugen und über `vars_files` einzubinden; alternativ `include_vars` oder `-e @generated.json` nutzen. Pfade und Datenstruktur müssen übereinstimmen. Beispiel für `generated.json`: `{"app_config": {"web_port": 8080}}`; das Template greift dann auf `{{ app_config.web_port }}` zu.

### Teil 5 – Aussagekräftige Tests
Vor dem ersten Lauf Connectivity und Syntax prüfen. Nach dem ersten Lauf die tatsächlich erzeugten Dateien und Dienste auf den Zielhosts prüfen. Der zweite Lauf soll ohne geänderte Eingabedaten keine unnötigen Änderungen melden. Für den Fehlerfall beispielsweise eine erforderliche Variable gezielt entfernen und die erwartete Fehlermeldung dokumentieren; danach den Ausgangszustand wiederherstellen.

### Abschlussreflexion – sechs Antworten
1. **Shell:** Starten und Verknüpfen vorhandener lokaler Programme; keine komplexe fachliche Datenverarbeitung
2. **Python:** JSON-Parsing, Validierung, Transformation und Ableitung kundenspezifischer Werte
3. **Ansible:** Inventory-basierte Paket-, Datei-, Template- und Dienstverwaltung mit definierten Zielzuständen
4. **Idempotenz:** Wiederholte Anwendung derselben Konfiguration ohne unnötige Änderungen, einschließlich Template und Handler
5. **100 Hosts:** Gruppen, Variablen, Rollen und parallele Ausführung gewinnen an Bedeutung; zusätzlich Batch-Rollout (`serial`), Fehlerbehandlung und Monitoring berücksichtigen
6. **Professionalisierung:** Tests in CI, Versions- und Secret-Management, Linting, Rollback-Strategie, Logging und dokumentierte Freigaben

## Durchgängige Referenzimplementierung – Python → Ansible

Die folgende Variante konkretisiert die vorherige Referenzarchitektur. Sie ist ein **alternativer, zusammenhängender Lösungsweg**; das oben gezeigte listenbasierte `generated.json` wird hier durch ein Host-Dictionary ersetzt, damit Ansible die Werte eindeutig pro Host zuordnen kann. Die Hosts `web01`, `web02` und `db01` sind an die Trainingsumgebung anzupassen.

### Eingabedatei `servers.json`

```json
[
  {"name": "web01", "role": "web"},
  {"name": "web02", "role": "web"},
  {"name": "db01", "role": "db"}
]
```

### `generate_config.py`

```python
import json
from pathlib import Path

source = json.loads(Path("servers.json").read_text(encoding="utf-8"))
if not isinstance(source, list):
    raise ValueError("servers.json muss eine Liste enthalten")

generated = {}
for item in source:
    if not isinstance(item, dict) or not isinstance(item.get("name"), str) or not item["name"].strip():
        raise ValueError("Jeder Server benötigt einen nichtleeren Namen")
    if item.get("role") not in {"web", "db"}:
        raise ValueError("Unbekannte oder fehlende Serverrolle")
    name = item["name"]
    if name in generated:
        raise ValueError(f"Doppelter Hostname: {name}")
    generated[name] = {"service_port": 8080 if item["role"] == "web" else 5432,
                       "server_role": item["role"]}

Path("generated.json").write_text(
    json.dumps({"generated_servers": generated}, indent=2) + "\n",
    encoding="utf-8"
)
```

### `ansible/inventory.yml`

```yaml
all:
  children:
    web:
      hosts:
        web01:
        web02:
    db:
      hosts:
        db01:
```

### `ansible/group_vars/web.yml` und `ansible/group_vars/db.yml`

```yaml
# web.yml
training_package: nginx
```

```yaml
# db.yml
training_package: tree
```

Die Beispielpakete `nginx` (Webserver) und `tree` (Datenbankgruppe) müssen für die Zielumgebung freigegeben und verfügbar sein. Für die Dienstverwaltung wird auf Webhosts ein systemd-fähiger Linux-Host mit `nginx` vorausgesetzt. Die Hostvariablen können optional in `host_vars/` überschrieben werden.

### `ansible/site.yml`

```yaml
- name: Trainingskonfiguration
  hosts: all
  gather_facts: true
  become: true
  vars_files:
    - ../generated.json
  roles:
    - app_config
```

### `ansible/roles/app_config/defaults/main.yml`

```yaml
training_package: tree
training_config_path: /tmp/training-app.conf
```

### `ansible/roles/app_config/tasks/main.yml`

```yaml
- name: Hostdaten müssen vorhanden sein
  ansible.builtin.assert:
    that:
      - inventory_hostname in generated_servers
    fail_msg: "Keine generierten Daten für diesen Inventory-Host"

- name: Freigegebenes Paket sicherstellen
  ansible.builtin.package:
    name: "{{ training_package }}"
    state: present

- name: Distribution für Diagnose protokollieren
  ansible.builtin.debug:
    msg: "Distribution: {{ ansible_facts['distribution'] }}"
  when: ansible_facts['distribution'] is defined

- name: Konfiguration rendern
  ansible.builtin.template:
    src: app.conf.j2
    dest: "{{ training_config_path }}"
    mode: "0644"
  notify: Konfiguration aktualisiert

- name: Webdienst starten und aktivieren
  ansible.builtin.service:
    name: nginx
    state: started
    enabled: true
  when: server_role == 'web'
  vars:
    server_role: "{{ generated_servers[inventory_hostname].server_role }}"
```

### `ansible/roles/app_config/templates/app.conf.j2`

```jinja2
hostname={{ inventory_hostname }}
role={{ generated_servers[inventory_hostname].server_role }}
port={{ generated_servers[inventory_hostname].service_port }}
```

### `ansible/roles/app_config/handlers/main.yml`

```yaml
- name: Konfiguration aktualisiert
  ansible.builtin.service:
    name: nginx
    state: restarted
  when: generated_servers[inventory_hostname].server_role == 'web'
```

Der Handler startet nur auf Webhosts den Dienst `nginx` neu, wenn sich das Template verändert hat. Auf DB-Hosts ist kein entsprechender Webdienst erforderlich. In Containern ohne systemd oder ohne verwaltbaren nginx-Dienst muss stattdessen ein für die Schulungsumgebung freigegebener Dienst verwendet werden.

### Integration und Nachweis

```bash
python3 generate_config.py
ansible-inventory -i ansible/inventory.yml --graph
ansible all -i ansible/inventory.yml -m ansible.builtin.ping
ansible-playbook -i ansible/inventory.yml ansible/site.yml --syntax-check
ansible-playbook -i ansible/inventory.yml ansible/site.yml
ansible-playbook -i ansible/inventory.yml ansible/site.yml
```

Die beiden Playbook-Läufe vergleichen: Im ersten Lauf können Paket und Template geändert werden, im zweiten sollte bei unveränderten Eingaben kein unnötiges `changed` entstehen. Als kontrollierten Fehlerfall `db01` aus `servers.json` entfernen, neu generieren und den erwarteten `assert`-Fehler für `db01` dokumentieren. Danach die Eingabe wiederherstellen. Die Musterlösung zeigt damit den vollständigen Datenfluss; die konkrete Dienstverwaltung muss entsprechend der freigegebenen Schulungsumgebung ergänzt werden.